In [1]:
import os
#os.environ['CUDA_VISIBLE_DEVICES']='1'

from matplotlib import pyplot as plt 
import numpy as np
import pandas as pd
import math
from pathlib import Path
import pickle

import jax
import jax.numpy as jnp
from tqdm import tqdm
import optax
import flax.linen as nn
from flax.metrics import tensorboard
from jax_resnet import resnet
from functools import partial
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

#import datasets
#import tensorflow_datasets as tfds

In [ ]:
"""
# Create param files if necessary (one .npy file = Om,S8,h,w0)

path_csv = '/net/GECO/pisco/SHEAR2COSMO/DBB_KMEANS_25/lhs_25.csv'
path_out = '/net/GECO/pisco/SHEAR2COSMO/DBB_KMEANS_25/lhs_25_params/'

data = pd.read_csv(path_csv)
data1 = data.iloc[:, 1:5]
table = np.array(data1)

n_seeds = 4          #number of seeds produced
n_maps = 104*n_seeds #number of maps per cosmology

if not os.path.exists(path_out):
  os.makedirs(path_out)

def export_table(table,n_maps):
    for i in range(len(table)):
        for k in range(n_maps):
            np.save(path_out+"param_id"+str(i).zfill(2)+"_seed"+str(k//104)+"_part"+str(k%104).zfill(3)+".npy",table[i])

export_table(table,n_maps)
"""

LOAD DATA

In [2]:
class DataLoader:
    def __init__(
        self,
        dir_path_image,
        dir_path_params,
        input_file_path,
        batch_size,
        mean_images,
        std_images,
        min_images,
        max_images,
        min_params,
        max_params,
        normalisation,
        params_bounds,
        shuffle,
        seeds,
        train_ratio,
        val_ratio,
        test_ratio
    ):
        self.dir_path_image = Path(dir_path_image)
        self.dir_path_params = Path(dir_path_params)
        self.input_file_path = input_file_path
        self.batch_size = batch_size
        self.min_images = min_images
        self.max_images = max_images
        self.mean_images = mean_images
        self.std_images = std_images
        self.min_params = min_params
        self.max_params = max_params
        self.normalisation = normalisation
        self.params_bounds = params_bounds
        self.shuffle = shuffle
        self.seeds = seeds

        #Import files depending on seeds
        self.image_files = sorted([f for f in self.dir_path_image.glob("id*.npy") if any(f"seed{seed}_" in f.stem for seed in self.seeds)])
        self.params_files = sorted([f for f in self.dir_path_params.glob("param_*.npy") if any(f"seed{seed}_" in f.stem for seed in self.seeds)])

        #Create indices
        self.indices = np.arange(len(self.image_files))
        
        assert len(self.image_files) == len(self.params_files)

        # Shuffle indices before splitting
        if self.shuffle:
            np.random.shuffle(self.indices)

        # Split data
        assert round(train_ratio + val_ratio + test_ratio,2) == 1.0, "Ratios sum must be 1."
        total_count = len(self.indices)
        train_end = int(total_count * train_ratio)
        val_end = train_end + int(total_count * val_ratio)
        
        self.train_indices = self.indices[:train_end]
        self.val_indices = self.indices[train_end:val_end]
        self.test_indices = self.indices[val_end:]

    # load *.npy file
    @staticmethod
    def load_npy(file):
        return np.load(file)
    
    @staticmethod
    def load_txt(file):
        return np.loadtxt(file)

    @staticmethod
    def min_max(x, mini, maxi, inf=0, sup=1):
        return (sup - inf) * (x - mini) / (maxi - mini) + inf

    @staticmethod
    def sigmoid(x):
        scale = 5
        return 1 / (1 + jnp.exp(-scale * (x - 0.5)))

    @staticmethod
    def streching(x):
        power = 1 / 3
        return jnp.power(x, power)

    @staticmethod
    def logy(x):
        scale = 10
        return jnp.log1p(x * scale)

    @staticmethod
    def range_compression(x):
        return jnp.arcsinh(x / 3)

    def load_batch_data(self, batch_indices, data_files):
        if data_files == self.image_files :
            return jnp.array([self.load_npy(data_files[idx]) for idx in batch_indices])
        if data_files == self.params_files :
            # HERE TO SELECT PARAMETERS TO LOAD
            return jnp.array([self.load_npy(data_files[idx])[0:2] for idx in batch_indices]) # [0,1,2,3] for [Om,S8,h,w0]

    def mean_std(self, x):
        return (x - self.mean_images) / self.std_images

    def generate(self,dataset):
            if dataset == "train":
                self.indices = self.train_indices
            elif dataset == "val":
                self.indices = self.val_indices
            elif dataset == "test":
                self.indices = self.test_indices
            # randomises data sent by dataloader
            if self.shuffle:
                key_shuffle = jax.random.PRNGKey(seed=int(np.random.randint(0, 1e6)))
                self.indices = jax.random.permutation(key=key_shuffle, x=self.indices)

            for i in range(0, len(self.indices), self.batch_size):
                batch_indices = self.indices[i : i + self.batch_size]

                # load images
                x = self.load_batch_data(batch_indices, self.image_files)
                y = self.load_batch_data(batch_indices, self.params_files)

                # images normalisation
                if self.normalisation == "sigmoid":
                    x = self.min_max(x, mini=self.min_images, maxi=self.max_images)
                    x = self.sigmoid(x)
                elif self.normalisation == "streching":
                    x = self.min_max(x, mini=self.min_images, maxi=self.max_images)
                    x = self.streching(x)
                elif self.normalisation == "compress":
                    x = self.mean_std(x)
                    x = self.range_compression(x)
                elif self.normalisation == "logy":
                    x = self.min_max(x, mini=self.min_images, maxi=self.max_images)
                    x = self.logy(x)
                elif self.normalisation == "minmax":
                    x = self.min_max(x, mini=self.min_images, maxi=self.max_images)

                # params normalisation
                y_normalized = []
                for i in range(y.shape[1]):
                    y_normalized.append(self.min_max(
                        y[:, i],
                        mini=self.min_params[i],
                        maxi=self.max_params[i],
                    ))
                y_normalized = jnp.stack(y_normalized, axis=-1)

                yield jax.numpy.expand_dims(x, axis=-1), jax.numpy.array(y_normalized)
                #yield jax.numpy.expand_dims(x, axis=-1), jax.numpy.array(y)

In [3]:
dir_path_image_val = '/net/GECO/pisco/SHEAR2COSMO/DBB_LHS_25/cartesian/'
dir_path_params_val = '/net/GECO/pisco/SHEAR2COSMO/DBB_LHS_25/lhs_25_params/'
dir_path_image_train = '/net/GECO/pisco/SHEAR2COSMO/DBB_DENSITY_25/cartesian/'
dir_path_params_train = '/net/GECO/pisco/SHEAR2COSMO/DBB_DENSITY_25/lhs_25_params/'
batch_size = 64
mean_images = 7e-05
std_images = 0.01
min_images = -0.08
max_images = 0.08
min_params = [0.1, 0.6]
max_params = [0.55, 0.9]
normalisation = "minmax"  # "sigmoid", "streching", "compress", "logy", "minmax"
params_bounds = [0.,1.]
shuffle = True
seeds = [0,1,2]
train_ratio = 0.8
val_ratio = 0.2
test_ratio = 0

# TRAIN DATASET
data_loader_train = DataLoader(dir_path_image=dir_path_image_train,
    dir_path_params=dir_path_params_train,
    input_file_path=None,  # Non utilisé
    batch_size=batch_size,
    mean_images=mean_images,
    std_images=std_images,
    min_images=min_images,
    max_images=max_images,
    min_params=min_params,
    max_params=max_params,
    normalisation=normalisation,
    params_bounds=params_bounds,
    shuffle=shuffle,
    seeds = seeds,
    train_ratio = train_ratio,
    val_ratio = val_ratio,
    test_ratio = test_ratio,)

# VALIDATION DATASET
data_loader_val = DataLoader(dir_path_image=dir_path_image_val,
    dir_path_params=dir_path_params_val,
    input_file_path=None,  # Non utilisé
    batch_size=batch_size,
    mean_images=mean_images,
    std_images=std_images,
    min_images=min_images,
    max_images=max_images,
    min_params=min_params,
    max_params=max_params,
    normalisation=normalisation,
    params_bounds=params_bounds,
    shuffle=shuffle,
    seeds = seeds,
    train_ratio = train_ratio,
    val_ratio = val_ratio,
    test_ratio = test_ratio,)

#Test batches
for x, y in data_loader_train.generate(dataset="train"):
    print("Test batch:")
    print("x shape:", x.shape)
    print("y shape:", y.shape)
    break

for x, y in data_loader_train.generate(dataset="val"):
    print("Validation batch:")
    print("x shape:", x.shape)
    print("y shape:", y.shape)
    break


train_count = len(data_loader_train.train_indices)
val_count = len(data_loader_train.val_indices)
test_count = len(data_loader_train.test_indices)

print("train : ", train_count)
print("validation : ", val_count)
print("test : ", test_count)
print("total : ",train_count+val_count+test_count)
print(data_loader_train.train_indices)


Test batch:
x shape: (64, 128, 128, 1)
y shape: (64, 2)
Validation batch:
x shape: (64, 128, 128, 1)
y shape: (64, 2)
train :  6240
validation :  1560
test :  0
total :  7800
[1387 6922 4507 ... 6357 5328   57]


In [4]:
# Define hyperparameters and suffix
N_cosmo = 25
N_maps = 104*N_cosmo*len(seeds)
epochs = 250

# Define a scheduler for the learning rate
total_steps = int(epochs*N_maps/batch_size)

init_value = 2e-3
nb_batch_train = N_maps/batch_size

lr_scheduler = optax.piecewise_constant_schedule( init_value=init_value,
    boundaries_and_scales={
        int(nb_batch_train * 40): 0.5,
        int(nb_batch_train * 80): 0.6,
        int(nb_batch_train * 120): 0.6,
        int(nb_batch_train * 160): 0.6,
    },)


loss_func = 'MSE'
# Define a simple MSE regression loss
def loss_fn(params, batch):
    return jnp.mean(jnp.sum((model.apply(params, batch['x']) - batch['y'])**2,axis=-1))
    #return jnp.mean(jnp.sum(jnp.abs(model.apply(params, batch['x']) - batch['y']),axis=-1))
    #return jnp.mean(jnp.sum((jnp.log1p(model.apply(params, batch['x'])) - jnp.log1p(batch['y']))**2, axis=-1))


#case = 'cosmoslics_resnet18_loss'+loss_func+'_norm_1'+'_Nmaps'+str(N_maps)+'_lri'+str(lri)+'_lr2'+str(lr2)+'_step2_'+str(step_2_ep)+'_batch'+str(batch_size)+'_epochs'+str(epochs)
case = 'cosmoslics_resnet18_loss'+loss_func+'norm_1'+'_Nmaps'+str(N_maps)+'_lri'+str(init_value)+'_lrsteps'+str(3)+'_batch'+str(batch_size)+'_epochs'+str(epochs)
print(nb_batch_train)

121.875


In [5]:
# Define our compressor
ResNet = partial(resnet.ResNet18, n_classes=2) #2 parameters to guess

class NeuralCompressor(nn.Module):
  @nn.compact
  def __call__(self, x):
    return ResNet()(x)

In [6]:
# Random seed for sampling from posterior
seed = jax.random.PRNGKey(0)

# Let's instantiate and initialize the model
model = NeuralCompressor()
params = model.init(jax.random.PRNGKey(42), jnp.zeros(shape=[1, 128, 128,1]))

# Initializes an optimizer and optimizer state
optimizer = optax.adam(learning_rate=lr_scheduler)
opt_state = optimizer.init(params)


#####################################
#summary_writer = tensorboard.SummaryWriter(LOGS_dir+case)
#####################################

In [ ]:
# Test some batches

cutouts = []
specz = []

# Generate batches of data using DataLoader
for x, y in data_loader.generate("train"):
    cutouts.append(x)
    specz.append(y)

# Stack the batches to get the complete dataset
cutouts = np.vstack(cutouts)
specz = np.vstack(specz)

In [ ]:
print(np.shape(cutouts))
print(np.shape(specz))

In [ ]:
plt.figure(figsize=(15,3))
for i,b in enumerate(['Map']):
  plt.subplot(1,5,i+1)
  plt.imshow(cutouts[0,:,:]) #,vmin=-4,vmax=8
  plt.title(b)
  plt.axis('off')
  print(specz[0])

plt.figure(figsize=(15,3))
for i,b in enumerate(['Map']):
  plt.subplot(1,5,i+1)
  plt.imshow(cutouts[1,:,:]) #,vmin=-4,vmax=8
  plt.title(b)
  plt.axis('off')
  print(specz[1])

In [ ]:
# Let's evaluate the noise standard deviation in each band, and apply range 
# compression accordingly

from astropy.stats import mad_std
scaling = []

for i,b in enumerate(['i']):
  plt.hist(cutouts[...].flatten(),100, label=b,alpha=0.5,);
  sigma = mad_std(cutouts[...].flatten())
  scaling.append(sigma)
  #plt.axvline(sigma, color='C%d'%i,alpha=0.5)
  #plt.axvline(-sigma, color='C%d'%i,alpha=0.5)
plt.legend()

In [ ]:
# Let's have a look at this distribution if we rescale it between 0 and 1 using max and min SNR values 
# over the dataset and center it

min_Mapmap = np.min(cutouts[...].flatten())
max_Mapmap = np.max(cutouts[...].flatten())

for i,b in enumerate([ 'i']):
  #plt.hist(cutouts[...].flatten()/scaling[i],100, label=b,alpha=0.5,
  plt.hist((cutouts[...].flatten()-min_Mapmap)/(max_Mapmap-min_Mapmap),100, label=b,alpha=0.5,
           range=[0,1]);
plt.legend()
plt.yscale('log')

print(min_Mapmap)  

In [ ]:
# a common approach for range compression is to apply arcsinh to suppress the
# high amplitude values
#for i,b in enumerate(['i']):
#  plt.hist(np.arcsinh(cutouts[...].flatten()/scaling[i]/3),100,
#           label=b, alpha=0.5,
#           range=[-5,5]);
#plt.legend()
#plt.yscale('log')

In [ ]:
# we can have a look at individual postage stamps with or without this scaling
#plt.subplot(121)
#plt.imshow(cutouts[0,:,:]/scaling[0])
#plt.title('Before')
#plt.subplot(122)
#plt.imshow(np.arcsinh(cutouts[0,:,:]/scaling[0]/3))
#plt.title('After');

In [7]:
@jax.jit
def update_model(params,  opt_state, batch):
  # Computes the gradients of the model
  loss, grads = jax.value_and_grad(loss_fn)(params, batch)

  # Computes the weights updates and apply them
  updates, opt_state = optimizer.update(grads, opt_state)
  params = optax.apply_updates(params, updates)

  return params, opt_state, loss

In [9]:
# Generate train and validation datasets
train_data = list(data_loader_train.generate(dataset="train"))
val_data = list(data_loader_train.generate(dataset="val"))

In [10]:
#################################################################################################
########## MAIN LOOP
#################################################################################################

# Number of epochs
NUMBER_OF_EPOCHS = epochs
PATIENCE = 300 # Number of epochs without amelioration before stopping
epochs_count = 0

# Init lists to stack losses
train_losses = []
val_losses = []

# Init to keep best model, based on best validation loss
best_val_loss = float('inf')
best_params = None

# Loop over epochs
for epoch in tqdm(range(NUMBER_OF_EPOCHS),desc="Epochs", leave=True):

    epochs_count += 1
    
    # Stack losses
    list_train_losses, list_val_losses = [], []

    # Training dataset
    for x, y in tqdm(train_data, desc="Training", leave=False):
        batch_train = {'x': x, 'y': y}
        params, opt_state, loss = update_model(params, opt_state, batch_train)
        list_train_losses.append(loss)
        if math.isnan(loss):
            print("NaN spotted!")
            print(x)
            exit()

    # Validation dataset
    for x, y in tqdm(val_data, desc="Validation", leave=False):
        batch_val = {'x': x, 'y': y}
        val_loss = loss_fn(params, batch_val)
        list_val_losses.append(val_loss)

    # stacking losses for current epoch
    train_loss_mean = np.mean(list_train_losses)
    val_loss_mean = np.mean(list_val_losses)
    train_losses.append(train_loss_mean)
    val_losses.append(val_loss_mean)

    # Condition to save the best params
    if val_loss_mean < best_val_loss:
        best_val_loss = val_loss_mean
        best_params = params
        epochs_no_improve = 0  # Reset the counter if we get a new best loss
    else:
        epochs_no_improve += 1

    # Early stopping condition
    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping on epoch {epoch+1}")
        break

print("Training complete.")

Epochs:   0%|          | 0/250 [00:00<?, ?it/s]2024-07-24 12:17:38.140457: W external/xla/xla/service/gpu/gemm_fusion_autotuner.cc:777] Compiling 47 configs for 3 fusions on a single thread.
2024-07-24 12:18:06.719395: W external/xla/xla/service/gpu/gemm_fusion_autotuner.cc:777] Compiling 40 configs for 3 fusions on a single thread.


In [ ]:
# Plot losses
plt.figure(figsize=(10, 5))
plt.plot(np.arange(NUMBER_OF_EPOCHS), train_losses, label='Train Loss')
plt.plot(np.arange(NUMBER_OF_EPOCHS), val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss, batch size '+str(batch_size)+'')
plt.ylim(0,0.15)
plt.grid()
plt.show()

Test on different maps

In [ ]:
dir_path_image = '/net/GECO/pisco/SHEAR2COSMO/DBB_TEST_25/cartesian/'
dir_path_params = '/net/GECO/pisco/SHEAR2COSMO/DBB_TEST_25/lhs_25_params/'
mean_images = 7e-05
std_images = 0.01
min_images = -0.08
max_images = 0.08
min_params = [0.1, 0.6]
max_params = [0.55, 0.9]
normalisation = "minmax"  # "sigmoid", "streching", "compress", "logy", "minmax"
params_bounds = [0.,1.]
shuffle = True
seeds = [0]
train_ratio = 1.
val_ratio = 0.
test_ratio = 0.

data_loader_test = DataLoader(dir_path_image=dir_path_image,
    dir_path_params=dir_path_params,
    input_file_path=None,  # Non utilisé
    batch_size=batch_size,
    mean_images=mean_images,
    std_images=std_images,
    min_images=min_images,
    max_images=max_images,
    min_params=min_params,
    max_params=max_params,
    normalisation=normalisation,
    params_bounds=params_bounds,
    shuffle=shuffle,
    seeds = seeds,
    train_ratio = train_ratio,
    val_ratio = val_ratio,
    test_ratio = test_ratio,)

#Test batches
for x, y in data_loader_test.generate(dataset="train"):
    print("Test batch:")
    print("x shape:", x.shape)
    print("y shape:", y.shape)
    break

train_count = len(data_loader_test.train_indices)
val_count = len(data_loader_test.val_indices)
test_count = len(data_loader_test.test_indices)

print("train : ", train_count)
print("validation : ", val_count)
print("test : ", test_count)
print("total : ",train_count+val_count+test_count)
print(data_loader_test.train_indices)



In [ ]:
test_data1 =list(data_loader_test.generate(dataset="train"))

# Preds on test dataset
truth = []
preds = []

for x, y in tqdm(test_data1, desc="Test", leave=False):
    batch_test = {'x': x, 'y': y}
    predictions = model.apply(best_params, batch_test['x']) #apply best model
    truth.append(batch_test['y'])
    preds.append(predictions)

truth = np.concatenate(truth)
preds = np.concatenate(preds)


# Fit for each parameter
def inv_min_max(y, mini, maxi, inf=0, sup=1):
  """
  un-normalize parameters values
  """
  return mini + (y - inf) * (maxi - mini) / (sup - inf)

new_truth = []
for i in range(y.shape[1]):
    new_truth.append(inv_min_max(truth[:, i],mini=min_params[i],maxi=max_params[i]))
new_truth = jnp.stack(new_truth, axis=-1)

new_preds = []
for i in range(y.shape[1]):
    new_preds.append(inv_min_max(preds[:, i],mini=min_params[i],maxi=max_params[i]))
new_preds = jnp.stack(new_preds, axis=-1)


# Polyfit on truth/preds
poly_Om = np.polyfit(new_truth[:,0], new_preds[:,0], 1)
poly_S8 = np.polyfit(new_truth[:,1], new_preds[:,1], 1)

# Calculate mean bias for each parameter
mean_bias_Om = np.mean(new_preds[:, 0] - new_truth[:, 0])
mean_bias_S8 = np.mean(new_preds[:, 1] - new_truth[:, 1])

# Calculate residuals for each parameter
residuals_Om = new_preds[:, 0] - new_truth[:, 0]
residuals_S8 = new_preds[:, 1] - new_truth[:, 1]


# GET TRAINING DATA FOR FINAL PLOT
original = []
for x, y in tqdm(train_data, desc="Test", leave=False):
    batch_test = {'x': x, 'y': y}
    original.append(batch_test['y'])
original = np.concatenate(original)

new_original = []
for i in range(2):
    new_original.append(inv_min_max(original[:, i],mini=min_params[i],maxi=max_params[i]))
new_original = jnp.stack(new_original, axis=-1)

In [ ]:
# Tracer les résultats
plt.figure(figsize=(15, 15))

#
plt.subplot(221)
#t = np.linspace(-0.05,1.05)
t = np.linspace(0.1,0.6)
plt.plot(t,t, color='k')
plt.fill_between(t,t-0.05*0.3,t+0.05*0.3, color='gray',alpha=0.5)
plt.plot(t, poly_Om[1]+poly_Om[0]*t, color='C1')

plt.scatter(new_truth[:,0], new_preds[:,0], alpha=0.2, label='$\Omega_{\\rm m}$',color='C1')
plt.scatter(new_original[:,0],new_original[:,0], color='steelblue', label='Training CPs')
plt.xlabel('true')
plt.ylabel('pred')

plt.legend();
plt.title('$\Omega_{\\rm m}^{\\rm pred} = %0.4f \\times \Omega_{\\rm m}^{\\rm true} + %0.4f$'%(poly_Om[0],poly_Om[1]))
plt.ylim(0.1,0.6)
plt.xlim(0.1,0.6)
#plt.ylim(-0.05,1.05)
#plt.xlim(-0.05,1.05)

#
plt.subplot(222)

t = np.linspace(0.5,1)
plt.plot(t,t, color='k')
plt.fill_between(t,t-0.05*0.75,t+0.05*0.75, color='gray',alpha=0.5)
plt.plot(t, poly_S8[1]+poly_S8[0]*t, color='C3')

plt.scatter(new_truth[:,1], new_preds[:,1], alpha=0.2, label='$S_8$',color='C3')
plt.scatter(new_original[:,1],new_original[:,1], color='steelblue', label='Training CPs')
plt.xlabel('true')
plt.ylabel('pred')

plt.legend();
plt.title('$S_8^{\\rm pred} = %0.4f \\times S_8^{\\rm true} + %0.4f$'%(poly_S8[0],poly_S8[1]))
#plt.ylim(-0.05,1.05)
#plt.xlim(-0.05,1.05)
plt.ylim(0.5,1)
plt.xlim(0.5,1)


plt.subplot(223)
plt.axhline(mean_bias_Om, color='red', linestyle='--', label='Mean Bias')
plt.scatter(new_truth[:, 0], residuals_Om, alpha=0.2, label='Residuals', color='C1')
plt.xlabel('True $\Omega_m$')
plt.ylabel('Residual')
plt.legend()
plt.title('Mean Bias and Residuals for $\Omega_m$')

#
plt.subplot(224)
plt.axhline(mean_bias_S8, color='red', linestyle='--', label='Mean Bias')
plt.scatter(new_truth[:, 1], residuals_S8, alpha=0.2, label='Residuals', color='C3')
plt.xlabel('True $S_8$')
plt.ylabel('Residual')
plt.legend()
plt.title('Mean Bias and Residuals for $S_8$')

In [ ]:
param_names = ['Omega_m', 'S8']

# Loop for each parameter
for i, param_name in enumerate(param_names):
    # Computes metrics
    r2 = r2_score(new_truth[:, i], new_preds[:, i])
    mae = mean_absolute_error(new_truth[:, i], new_preds[:, i])
    mse = mean_squared_error(new_truth[:, i], new_preds[:, i])
    rmse = np.sqrt(mse)
    
    # Print metrics
    print(f'Parameter {param_name}:')
    print(f'  R^2 score: {r2:.3f}')
    print(f'  Mean Absolute Error: {mae:.3f}')
    print(f'  Mean Squared Error: {mse:.5f}')
    print(f'  Root Mean Squared Error: {rmse:.3f}')